# 🧠 SHAP Explainability

Explicabilidad de predicciones individuales y globales usando SHAP (SHapley Additive exPlanations).

**Qué hace esta notebook:**
- Carga un modelo entrenado y su feature engineering
- Computa valores SHAP para explicar predicciones
- Genera summary plot: qué features importan más globalmente
- Genera dependence plots: cómo afecta cada feature a la predicción
- Waterfall plots: explicación visual de un caso individual
- Análisis de los clientes más sospechosos: ¿por qué fueron marcados?

**Cuándo usarla:**
- Para entender qué está haciendo realmente el modelo (más allá del AUC)
- Para responder "¿por qué este cliente fue marcado como fraude?"
- Para detectar features que están "filtrando" el target (data leakage)
- Para comunicar resultados a equipos no técnicos

**Requisitos:**
- `pip install shap`
- Modelo entrenado (.pkl) + feature engineering (.pkl)
- Datos de ejemplo para computar SHAP (puede ser el conjunto de test o inferencia)

## 0. Configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN — paths, archivos y parámetros
# ============================================================
# Toda la configuración está acá. Para apuntar la notebook a otro
# proyecto o dataset, modificá estos valores (no hace falta tocar el
# resto de las celdas).

from pathlib import Path

# --- Proyecto y datos ---
PROJECT_PATH = Path('<<PROJECT_PATH>>')
OUTPUT_PATH = PROJECT_PATH / 'output'
VERSION = '<<VERSION>>'
TRAIN_DIR = '<<TRAIN_DIR>>'

# --- Artefactos de entrenamiento ---
MODEL_PATH = OUTPUT_PATH / VERSION / TRAIN_DIR / 'models' / 'model.pkl'
FE_PATH = OUTPUT_PATH / VERSION / TRAIN_DIR / 'models' / 'feature_engineering.pkl'

# --- Datos para explicar ---
# Usamos el dataset procesado de train porque incluye TODAS las columnas
# de entrada crudas (actividad, periodo, material, zona, etc.) que el pipeline
# de feature engineering necesita para transformar.
# Las probabilidades se calculan abajo usando el modelo sobre la muestra.
DATA_PATH = OUTPUT_PATH.parent / 'data' / 'processed' / VERSION / '<<DATASET_FILE>>'

# --- Entorno (detección automática Colab vs local) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Cuántas filas usar para computar SHAP (más = más lento pero más preciso)
N_SAMPLES = 2000

# Cuántos clientes investigar en detalle con waterfall plots
N_WATERFALL = 5

print(f'PROJECT_PATH : {PROJECT_PATH}')
print(f'MODEL_PATH   : {MODEL_PATH}')
print(f'FE_PATH      : {FE_PATH}')
print(f'DATA_PATH    : {DATA_PATH}')
print(f'IN_COLAB     : {IN_COLAB}')

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

from energizados.core.utils.integrity_pickle import load as safe_load

# Verificar archivos
for p in [MODEL_PATH, FE_PATH, DATA_PATH]:
    assert Path(p).exists(), f"No se encuentra: {p}"

print("✅ Configuración lista")


## 1. Carga del modelo y datos

Cargamos el modelo entrenado, el feature engineering (transformaciones), y los datos a explicar.

**Qué cargamos:**
- `feature_engineering.pkl`: pipeline de transformación (preprocesamiento + feature selection). Transforma datos crudos en features que el modelo entiende.
- `model.pkl`: el modelo entrenado (puede ser un adapter como `LGBMModelAdapter`).
- `predictions.csv`: datos ya transformados? Depende — si `include_input` estaba activado en inferencia, incluye los features. Si no, tenemos que aplicar FE manualmente.

> ⚠️ **Importante sobre `include_input`:** Esta notebook asume que el CSV de predicciones **NO** incluye los features transformados. Por eso aplicamos el FE pipeline a los datos crudos.

In [ ]:
# Cargar modelo
model = safe_load(MODEL_PATH)
print(f"Modelo: {type(model).__name__}")

# Cargar feature engineering
fe = safe_load(FE_PATH)
print(f"FE: {type(fe).__name__}")

# Cargar datos (auto-detecta formato por extensión)
_data_path = Path(DATA_PATH)
if _data_path.suffix == ".parquet":
    df_raw = pd.read_parquet(_data_path)
elif _data_path.suffix == ".csv":
    df_raw = pd.read_csv(_data_path)
else:
    raise ValueError(f"Formato no soportado: {_data_path.suffix}")
print(f"Datos: {df_raw.shape}")

# Identificar columnas
CONSUMPTION_COLS = [c for c in df_raw.columns if c.endswith("_anterior")]
META_COLS = [c for c in ["cliente", "geo_region", "probability"] if c in df_raw.columns]
print(f"  Columnas de consumo: {len(CONSUMPTION_COLS)}")
print(f"  Columnas meta: {META_COLS}")
df_raw.head(3)


## 2. Preparar features para SHAP

Aplicamos el pipeline de feature engineering para obtener la matriz de features que el modelo realmente usa.

**Concepto clave:** SHAP explica las predicciones en función de los **features transformados** (los que entran al modelo), no de los datos crudos. Por eso necesitamos aplicar el FE pipeline primero.

> Si trabajás con datos que ya están transformados (ej. `include_input: true` en inferencia), podés saltear este paso y usar las columnas directamente.

In [ ]:
# Aplicar feature engineering a los datos
# El FE pipeline necesita TODAS las columnas de entrada crudas (cliente, geo_region,
# periodo, actividad, material, etc.). Solo excluimos las columnas de salida/meta
# (probability, target, prediction); sin el resto el FE falla por columnas faltantes.
_non_feature = [c for c in ['probability', 'target', 'prediction'] if c in df_raw.columns]
feature_cols_raw = [c for c in df_raw.columns if c not in _non_feature]

# Aplicar FE
try:
    X_transformed = fe.transform(df_raw[feature_cols_raw])
    print(f"✅ FE aplicado: {X_transformed.shape}")
except Exception as e:
    print(f"⚠️  El FE pipeline requiere columnas específicas. Error: {e}")
    print("   Usando datos crudos como features (el modelo puede no coincidir)")
    X_transformed = df_raw[feature_cols_raw].select_dtypes(include=[np.number])

# Obtener nombres de features
if hasattr(X_transformed, 'columns'):
    feature_names = list(X_transformed.columns)
elif hasattr(X_transformed, 'get_feature_names_out'):
    feature_names = list(X_transformed.get_feature_names_out())
else:
    feature_names = [f"feature_{i}" for i in range(X_transformed.shape[1])]

print(f"Features totales: {len(feature_names)}")
print(f"Primeros 10 features: {feature_names[:10]}")

In [ ]:
# Tomar una muestra para SHAP (cómputo intensivo)
n_available = len(X_transformed)
n_sample = min(N_SAMPLES, n_available)

# Calcular probabilidades en todo el dataset para poder hacer sampling estratificado
# (la columna 'probability' no existe en el dataset de train, la generamos nosotros)
print(f"Calculando probabilidades para {n_available:,} filas...", flush=True)
_proba = model.predict_proba(X_transformed)
# El wrapper puede devolver (n, 2) o (n,) dependiendo del adapter
if _proba.ndim == 2:
    _proba = _proba[:, 1]
df_raw["probability"] = _proba

if n_sample < n_available:
    # Samplear con prioridad en alta probabilidad para ver casos interesantes
    high_prob_idx = df_raw["probability"].nlargest(n_sample // 2).index
    random_idx = df_raw.sample(n_sample // 2, random_state=42).index
    sample_idx = high_prob_idx.union(random_idx)[:n_sample]
else:
    sample_idx = df_raw.index[:n_sample]

X_sample = X_transformed.iloc[sample_idx] if hasattr(X_transformed, 'iloc') else X_transformed[sample_idx]
X_sample = pd.DataFrame(X_sample, columns=feature_names)
probas_sample = df_raw.loc[sample_idx, "probability"].values

print(f"Muestra para SHAP: {len(X_sample):,} filas × {X_sample.shape[1]} features")
print(f"Rango probabilidad en la muestra: [{probas_sample.min():.4f}, {probas_sample.max():.4f}]")


## 3. Computar valores SHAP

SHAP asigna a cada feature un valor que representa cuánto contribuyó a la predicción de ese caso, comparado con la predicción promedio.

**Qué tarda y por qué:**
- **TreeExplainer** (LightGBM, CatBoost): rápido, exacto. Segundos.
- **KernelExplainer** (modelos genéricos, NN, ensembles): lento, aproximado. Minutos.

Si ves "Using KernelExplainer", considerá limitar más `N_SAMPLES` o usar un modelo basado en árboles para explainability.

**SHAP values interpretación:**
- Valor positivo = la feature empuja la predicción **hacia arriba** (más probabilidad de fraude).
- Valor negativo = la feature empuja **hacia abajo** (menos probabilidad de fraude).
- Magnitud = importancia de esa feature para ese caso específico.

In [ ]:
%%time
print("Computando SHAP values...")

# Intentar usar TreeExplainer directamente (más rápido que la clase wrapper)
raw_model = None
if hasattr(model, 'get_raw_model'):
    raw_model = model.get_raw_model()
    if isinstance(raw_model, dict):
        raw_model = next(iter(raw_model.values()))
elif hasattr(model, 'predict_proba'):
    raw_model = model

model_type = type(raw_model).__name__ if raw_model is not None else "unknown"
print(f"Modelo raw: {model_type}")

# Seleccionar el explicador adecuado
if model_type in ("LGBMClassifier", "LGBMRegressor", "CatBoostClassifier", "CatBoostRegressor"):
    print("→ Usando TreeExplainer (rápido y exacto)")
    explainer = shap.TreeExplainer(raw_model)
    # np.asarray evita el mismatch de categorical_feature de LightGBM con DataFrame
    shap_values = explainer.shap_values(np.asarray(X_sample))
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # para clasificación binaria, clase positiva
    expected_value = explainer.expected_value
    if isinstance(expected_value, (list, np.ndarray)):
        expected_value = expected_value[1] if len(expected_value) > 1 else expected_value[0]
else:
    print(f"→ Modelo {model_type}: usando KernelExplainer (más lento, aproximado)")
    # Usar una muestra de background
    background = shap.kmeans(X_sample, min(50, len(X_sample)))
    predict_fn = raw_model.predict_proba if hasattr(raw_model, 'predict_proba') else raw_model.predict
    explainer = shap.KernelExplainer(predict_fn, background)
    shap_values = explainer.shap_values(X_sample, nsamples=100)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    expected_value = explainer.expected_value
    if isinstance(expected_value, (list, np.ndarray)):
        expected_value = expected_value[0]

print(f"✅ SHAP values: {shap_values.shape}")
print(f"   Expected value (predicción base): {expected_value:.4f}")

## 4. Summary Plot: importancia global de features

El gráfico más importante de SHAP. Muestra qué features impactan más las predicciones.

**Cómo leerlo:**
- **Eje Y:** features ordenadas por importancia (la más importante arriba).
- **Eje X:** valor SHAP. Puntos a la derecha = empujan la predicción hacia "fraude". Puntos a la izquierda = empujan hacia "no fraude".
- **Color:** valor de la feature. Rojo = valor alto de la feature, Azul = valor bajo.
- **Patrón revelador:** si los puntos rojos están consistentemente a la derecha, significa que valores ALTOS de esa feature aumentan la probabilidad de fraude.

**Señales de alerta:**
- Una feature en el top 3 que no debería ser predictiva → posible data leakage.
- Features con muy poca dispersión horizontal → no son útiles, el modelo las ignora.
- Features con rojos a la derecha Y azules a la izquierda → relación monótona con el target (buena señal).
- `if_score` muy arriba → el modelo se apoya mucho en el Isolation Forest (esperable).
- Features de consumo con poco impacto → el modelo no está usando la señal de consumo (revisar).

In [ ]:
# Summary plot
fig, ax = plt.subplots(figsize=(12, max(8, len(feature_names) * 0.3)))
shap.summary_plot(
    shap_values, X_sample, feature_names=feature_names,
    max_display=25, show=False
)
plt.title("SHAP Summary Plot — Importancia global de features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot: importancia media (más simple de interpretar)
fig, ax = plt.subplots(figsize=(10, max(6, len(feature_names) * 0.25)))
shap.summary_plot(
    shap_values, X_sample, feature_names=feature_names,
    plot_type="bar", max_display=20, show=False
)
plt.title("SHAP Feature Importance (media |SHAP|)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Top features: tabla de importancia

Versión numérica del summary plot para incluir en informes.

**Columnas:**
- `mean_abs_shap`: importancia media (promedio del valor absoluto del SHAP). Más alto = más impacto.
- `mean_shap`: dirección del impacto. Positivo = tiende a aumentar probabilidad de fraude. Negativo = tiende a disminuirla.
- `shap_std`: cuánto varía el impacto entre casos. Alto = la feature afecta de manera muy distinta a diferentes clientes.

In [ ]:
# Tabla de importancia
importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    "mean_shap": shap_values.mean(axis=0),
    "shap_std": shap_values.std(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print("=== Top 20 features por importancia SHAP ===")
try:
    display(importance_df.head(20).style
        .background_gradient(subset=["mean_abs_shap"], cmap="Blues")
        .format({"mean_abs_shap": "{:.6f}", "mean_shap": "{:.6f}", "shap_std": "{:.6f}"}))
except (AttributeError, ImportError):
    # Fallback sin styling si jinja2 no está instalado
    print(importance_df.head(20).to_string(index=False))

# Destacar features con dirección inesperada
print("\n=== Features con dirección potencialmente contra-intuitiva ===")
suspicious = importance_df[
    (importance_df["mean_shap"] > 0) & (importance_df["feature"].str.contains("consumo|anterior", case=False))
]
if len(suspicious) > 0:
    print("⚠️  Features de consumo con SHAP positivo (más consumo → más fraude):")
    try:
        display(suspicious)
    except (AttributeError, ImportError):
        print(suspicious.to_string(index=False))
else:
    print("✅ Ningún feature de consumo tiene SHAP positivo inesperado")


## 6. Dependence Plots: cómo una feature afecta la predicción

Cada gráfico muestra cómo cambia el valor SHAP a medida que cambia el valor de la feature.

**Cómo leerlo:**
- **Eje X:** valor real de la feature (ej. consumo en kWh).
- **Eje Y:** valor SHAP (impacto en la predicción).
- **Color:** valor de otra feature con la que interactúa (SHAP elige automáticamente la interacción más fuerte).
- **Patrón deseable:** relación suave y monótona. Ej: a menor consumo, mayor SHAP (más fraude).
- **Patrón ruidoso:** nube de puntos sin forma clara. La feature tiene efectos inconsistentes.
- **Patrón en U o V:** la feature tiene un punto óptimo. Ej: consumo muy bajo O muy alto → fraude.
- **Puntos del mismo color agrupados:** la feature interactúa fuertemente con la feature del color.

In [ ]:
# Dependence plots para las top 6 features
top6_features = importance_df.head(6)["feature"].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feat in zip(axes.flatten(), top6_features):
    shap.dependence_plot(
        feat, shap_values, X_sample, feature_names=feature_names,
        ax=ax, show=False
    )
    ax.set_title(feat, fontsize=11, fontweight="bold")

plt.suptitle("SHAP Dependence Plots — Top 6 Features", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 7. Waterfall Plots: explicación de casos individuales

Para un cliente específico, mostramos exactamente cómo cada feature contribuyó a su predicción.

**Cómo leerlo:**
- `E[f(x)]` = predicción base (promedio del modelo sobre todos los clientes).
- `f(x)` = predicción final para este cliente.
- Cada barra es una feature: roja = empuja hacia arriba (más fraude), azul = empuja hacia abajo (menos fraude).
- El tamaño de la barra = magnitud del impacto.

**Ejemplo de lectura:**
> "Este cliente tiene probabilidad 0.92 de fraude. El modelo partió de una base de 0.81 y:
> - `if_score` alto sumó +0.06 (anomalía detectada por Isolation Forest)
> - `consumo_1_anterior` bajo sumó +0.03 (poco consumo → sospechoso)
> - `zscore_last_vs_history` alto sumó +0.02 (caída abrupta vs histórico)"

Esto es lo que le mostrás a un inspector o a un gerente para justificar una inspección.

In [ ]:
# Waterfall plots para los N clientes más sospechosos de la muestra
top_indices = np.argsort(probas_sample)[-N_WATERFALL:][::-1]

for rank, idx in enumerate(top_indices, 1):
    proba = probas_sample[idx]
    cliente_id = df_raw.loc[sample_idx[idx], "cliente"] if "cliente" in df_raw.columns else f"row_{idx}"
    region = df_raw.loc[sample_idx[idx], "geo_region"] if "geo_region" in df_raw.columns else "?"
    
    print(f"\n{'='*70}")
    print(f"#{rank} | Cliente: {cliente_id} | Región: {region} | Probabilidad: {proba:.6f}")
    print(f"{'='*70}")
    
    fig, ax = plt.subplots(figsize=(10, max(5, min(20, len(feature_names)) * 0.3)))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[idx],
            base_values=expected_value,
            data=X_sample.iloc[idx].values,
            feature_names=feature_names,
        ),
        max_display=15,
        show=False,
    )
    plt.title(f"Waterfall — Cliente {cliente_id} (prob={proba:.4f})", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 8. Comparación: cliente de alta probabilidad vs baja probabilidad

Para entender qué diferencia a un cliente marcado como fraude de uno no marcado, comparamos sus waterfall plots lado a lado.

**Qué mirar:**
- ¿Son las mismas features las que importan en ambos casos? Si no, el modelo usa señales distintas según el perfil del cliente.
- ¿La diferencia principal está en features de consumo o en features categóricas? Si son categóricas (ej. `geo_region`), el modelo puede estar sesgado geográficamente.
- ¿El cliente de baja probabilidad tiene features que "lo salvan" (barras azules grandes)? Eso es lo que evita que sea marcado.

In [ ]:
# Comparar un cliente de alta probabilidad con uno de baja probabilidad
idx_high = np.argmax(probas_sample)
idx_low = np.argmin(probas_sample)

fig, axes = plt.subplots(1, 2, figsize=(20, max(6, len(feature_names) * 0.3)))

for ax, idx, label in [
    (axes[0], idx_high, f"Alta probabilidad ({probas_sample[idx_high]:.4f})"),
    (axes[1], idx_low, f"Baja probabilidad ({probas_sample[idx_low]:.4f})"),
]:
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[idx],
            base_values=expected_value,
            data=X_sample.iloc[idx].values,
            feature_names=feature_names,
        ),
        max_display=15, show=False,
    )
    ax.set_title(label, fontsize=12, fontweight="bold")

plt.suptitle("Comparación: Alta vs Baja probabilidad de fraude", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 9. Heatmap: SHAP values de los clientes más sospechosos

Una matriz donde cada fila es un cliente (top 30 más sospechosos) y cada columna es una feature (top 15 más importantes).

**Qué mirar:**
- **Columnas consistentemente rojas:** features que empujan a todos los sospechosos hacia arriba. Son los drivers principales del modelo.
- **Filas con patrón muy distinto:** clientes que son marcados por razones diferentes al resto. Ej: la mayoría son marcados por `if_score`, pero este cliente es marcado por `consumo_cero`.
- **Columnas azules en sospechosos:** features que "salvan" a estos clientes de tener aún mayor probabilidad.

In [ ]:
# Heatmap: top 30 clientes × top 15 features
n_clients_heatmap = min(30, len(X_sample))
n_features_heatmap = min(15, len(feature_names))

top_features_idx = np.argsort(np.abs(shap_values).mean(axis=0))[-n_features_heatmap:][::-1]
top_clients_idx = np.argsort(probas_sample)[-n_clients_heatmap:][::-1]

heatmap_data = pd.DataFrame(
    shap_values[top_clients_idx][:, top_features_idx],
    columns=[feature_names[i] for i in top_features_idx],
    index=[f"#{i+1} ({probas_sample[idx]:.3f})" for i, idx in enumerate(top_clients_idx)],
)

fig, ax = plt.subplots(figsize=(max(10, n_features_heatmap * 1), max(8, n_clients_heatmap * 0.35)))
sns.heatmap(heatmap_data, cmap="RdBu_r", center=0, annot=False, ax=ax,
            cbar_kws={"label": "SHAP value"})
ax.set_title(f"SHAP Values: Top {n_clients_heatmap} clientes × Top {n_features_heatmap} features",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Cliente (probabilidad)")
ax.set_xlabel("Feature")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 10. Export

Exportamos la tabla de importancia SHAP para incluir en informes.

In [ ]:
export_dir = Path(DATA_PATH).parent / "explainability"
export_dir.mkdir(exist_ok=True)

# Tabla de importancia
importance_df.to_csv(export_dir / "shap_importance.csv", index=False)
print(f"✅ {export_dir / 'shap_importance.csv'}")

# SHAP values completos (para análisis posteriores)
shap_df = pd.DataFrame(shap_values, columns=feature_names)
shap_df.index = sample_idx
shap_df.to_parquet(export_dir / "shap_values.parquet")
print(f"✅ {export_dir / 'shap_values.parquet'}")

print(f"\nArchivos exportados a: {export_dir.resolve()}")

---
## Notas

- **SHAP es intensivo en cómputo.** Para datasets grandes (> 50k filas), usá `N_SAMPLES` bajo (500-2000) para el análisis exploratorio.
- **TreeExplainer es mucho más rápido** que KernelExplainer. Si usás LGBM o CatBoost, SHAP vuela.
- **Los SHAP values se calculan sobre features transformadas.** Si querés explicar en términos de features originales (ej. `actividad` en vez de `actividad_prob`), necesitás mapear hacia atrás las transformaciones.
- Si `shap` no está instalado: `pip install shap`
- Para modelos muy grandes o ensembles, considerá usar solo el summary plot (menos intensivo que los waterfall plots individuales).